# Practical 4: Text Generation using an LSTM

**Aim:** To build a character-level LSTM network on the Shakespeare corpus to generate text, and compare it with the SimpleRNN of Practical 3.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Validation accuracy ~47%, perplexity ~6.8.; LSTM's gated cell state mitigates the vanishing gradient.

## Cell 1 - Import libraries

**What it does:**
- NumPy for arrays, Matplotlib for graphs, Keras for the network.

*Expected output (illustrative):* TensorFlow version: 2.x.x

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## Cell 2 - Download and read the Shakespeare text

**What it does:**
- Downloads the Tiny Shakespeare corpus (~1.1 million characters) once and reads it as text.
- `.lower()` converts everything to lowercase, which reduces the vocabulary to about 38-39 characters (matching your PDF's '~38 chars').
- If your lab provides its own text file, replace `path` with that file's path.

*Expected output (illustrative):* Total characters: 1115394, followed by the first lines of the play ('first citizen: before we proceed any further, hear me speak...').

In [ ]:
path = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")

text = open(path, "rb").read().decode("utf-8").lower()

print("Total characters:", len(text))
print(text[:250])

## Cell 3 - Build the character vocabulary

**What it does:**
- `chars` is the sorted list of unique characters; `vocab_size` is how many there are (the number of output classes).
- `char2idx` maps character -> number; `idx2char` maps number -> character. Neural networks need numbers, not letters.
- `encoded` is the whole text converted to integers.

*Expected output (illustrative):* Vocabulary size: about 38-39, then the list of characters (newline, space, punctuation, digits 3 and $, a-z).

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)

char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)

encoded = np.array([char2idx[c] for c in text], dtype="int32")

print("Vocabulary size:", vocab_size)
print(chars)

## Cell 4 - Create 40-character input windows (stride 3)

**What it does:**
- A window of **40 characters** is the input (`X`); the **41st character** is the target (`y`) - this is next-character prediction.
- `step = 3` moves the window 3 characters at a time, giving ~372,000 training examples while keeping memory low.
- Samples are shuffled so the validation split (taken from the end of the arrays) contains a mixture of the whole book, not just the last scenes.
- Because windows overlap, validation windows share text with training windows; the validation figures are therefore a little optimistic.

*Expected output (illustrative):* X: (about 371,785, 40) y: (about 371,785,)

In [ ]:
maxlen = 40
step = 3

starts = np.arange(0, len(encoded) - maxlen, step)
X = np.array([encoded[s:s + maxlen] for s in starts], dtype="int32")
y = encoded[starts + maxlen]

order = np.random.RandomState(42).permutation(len(X))
X, y = X[order], y[order]

print("X:", X.shape, "y:", y.shape)

## Cell 5 - Build the model

**What it does:**
- `Embedding(vocab_size, 64)` turns each character index into a learned 64-number vector (denser and more meaningful than one-hot).
- `LSTM(256)` is an RNN with input, forget and output gates and a cell state, so information can flow across many steps. It has 4x the parameters of a SimpleRNN of the same size.
- `Dense(vocab_size, softmax)` outputs a probability for every possible next character.

*Expected output (illustrative):* Embedding (40, 64) ~2.5k params; LSTM (256) = 4*256*(64+256+1) = 328,704 params; Dense (vocab) ~10k params. Total ~341k (with vocab = 39).

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(maxlen,)),
    layers.Embedding(vocab_size, 64),
    layers.LSTM(256),
    layers.Dense(vocab_size, activation="softmax"),
])

model.summary()

## Cell 6 - Compile and train

**What it does:**
- Labels `y` are integers (the index of the next character), so `sparse_categorical_crossentropy` is used.
- `batch_size=256` makes training faster on this large dataset; 10 epochs is a reasonable lab setting.
- LSTM is roughly 3x slower per epoch than SimpleRNN (from your PDF) because of its extra gates.

*Expected output (illustrative):* 10 epochs of logs. Validation accuracy approaches ~0.46 (SimpleRNN) / ~0.47 (LSTM); loss ~1.9.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history = model.fit(
    X, y,
    epochs=10,
    batch_size=256,
    validation_split=0.1,
)

## Cell 7 - Training curves and perplexity

**What it does:**
- **Perplexity = exp(loss)**. It can be read as 'the model is as unsure as if choosing between this many equally likely characters'. Lower is better.
- Your PDF's expected values: accuracy ~47%, perplexity ~6.8.

*Expected output (illustrative):* Validation perplexity about 6.8 (expected from your PDF).

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

val_loss = history.history["val_loss"][-1]
print("Validation loss      :", round(val_loss, 4))
print("Validation perplexity:", round(float(np.exp(val_loss)), 2))

## Cell 8 - Temperature sampling and generation functions

**What it does:**
- **Temperature** divides the log-probabilities before re-normalising: low (0.2) = safe, repetitive, coherent text; 1.0 = original distribution; high = random, creative but messy.
- `np.random.multinomial` randomly draws one character according to the probabilities (sampling) instead of always taking the most likely one.
- `generate_text` is the **autoregressive loop**: predict a character, append it, slide the 40-character window forward by one, and repeat - the model's own output becomes its next input.

*Expected output (illustrative):* No output (functions are defined).

In [ ]:
def sample(probs, temperature=1.0):
    probs = np.asarray(probs).astype("float64")
    logits = np.log(probs + 1e-9) / temperature
    exp_logits = np.exp(logits - np.max(logits))
    probs = exp_logits / np.sum(exp_logits)
    return int(np.argmax(np.random.multinomial(1, probs, 1)))


def generate_text(seed, length=400, temperature=1.0):
    window = [char2idx[c] for c in seed]
    result = seed
    for _ in range(length):
        x = np.array([window], dtype="int32")
        probs = model.predict(x, verbose=0)[0]
        next_id = sample(probs, temperature)
        result += idx2char[next_id]
        window = window[1:] + [next_id]
    return result

## Cell 9 - Generate text at temperatures 0.2, 0.5 and 1.0

**What it does:**
- A random 40-character piece of the original text is used as the seed.
- The same seed is used with three temperatures to compare creativity vs coherence.

*Expected output (illustrative):* Three blocks of 400 generated characters; compared with SimpleRNN the text is slightly more coherent over longer spans (from your PDF) but still contains invented words.

In [ ]:
start = np.random.randint(0, len(text) - maxlen - 1)
seed = text[start:start + maxlen]
print("SEED:", repr(seed))

for temp in (0.2, 0.5, 1.0):
    print("\n----- temperature =", temp, "-----")
    print(generate_text(seed, length=400, temperature=temp))

## Cell 10 - Save the model

**What it does:**
- Saves the trained network so you can generate text later without retraining.

*Expected output (illustrative):* Model saved.

In [ ]:
model.save("shakespeare_lstm.keras")
print("Model saved.")

## Result / Conclusion

The LSTM text generator reached ~47% validation accuracy (perplexity ~6.8), similar to the SimpleRNN in metrics but producing slightly more coherent long-range text at about three times the training cost.